In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from pysr import PySRRegressor

In [ ]:
column_names = ['CuCl^-1', 'Cu^+2', 'Cl^-1', 'Na^+1', 'K^+1', 'NO3^-1', 'Fe(CN)6^-4', 'Fe(CN)6^-3', 'T', 
               'pred_eta', 'pred_rho', 'pred_Aw', 'sigma_eta', 'sigma_rho', 'sigma_Aw']
#column_names = ['CuCl^-1', 'Cu^+2', 'Cl^-1', 'Na^+1', 'K^+1', 'NO3^-1', 'Fe(CN)6^-4', 'Fe(CN)6^-3', 'T', 
#               'pred_Cp', 'sigma_Cp']

df = pd.read_csv('Data/synthetic_Eta_rho_Aw.csv', names=column_names, header=0)
#df = pd.read_csv('Data/synthetic_Cp.csv', names=column_names, header=0)

df = df.drop(columns=['sigma_eta', 'sigma_rho', 'sigma_Aw'])
#df = df.drop(columns=['sigma_Cp'])

df.describe()

In [ ]:
X = pd.DataFrame(df, columns=['CuCl^-1', 'Cu^+2', 'Cl^-1', 'Na^+1', 'K^+1', 'NO3^-1', 'Fe(CN)6^-4', 'Fe(CN)6^-3', 'T'])
y = pd.DataFrame(df, columns=['pred_eta'])

X = X.rename(columns={'CuCl^-1': 'x1','Cu^+2': 'x2','Cl^-1': 'x3','Na^+1': 'x4','K^+1': 'x5','NO3^-1': 'x6','Fe(CN)6^-4': 'x7', 
                      'Fe(CN)6^-3': 'x8', 'T': 'x9'})
y = y.rename(columns={'pred_eta': 'y'})

In [ ]:
for i in range(1,41):
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.80, test_size = 0.20, random_state = i)
    
    model = PySRRegressor(
        procs=8,
        populations=16,
        population_size=1000,
        ncycles_per_iteration=200,
        niterations=100,
        maxsize=30,
        maxdepth=10,
        binary_operators=["*", "^", "+", "-", "/"], 
        unary_operators=["exp", "log"],
        complexity_of_variables=2,
        nested_constraints={"exp": { "exp": 0, "log":0},"log": { "exp": 0, "log":0}},
        progress=True,
        weight_randomize=0.1,
        cluster_manager=None,
        precision=64,
        turbo=True,
        julia_project=None,
        update=False,
        constraints={"^": (2, 1)})
    model.fit(X_train,y_train)
    equations_df = pd.DataFrame(model.equations_)
    equations_df.to_csv(f"PySR_Results\\eta_{i}.csv", index=False)